## 🚀 Multi-Agent Debate System with LangChain

This notebook implements a multi-agent debate system where two AI agents argue for and against a given topic, and a third AI agent judges the debate.

### 📦 Setup and Installation

First, we'll install the necessary LangChain libraries for interacting with Google's Gemini models.

In [9]:
# Install required libraries
%pip install --quiet langchain-core langchain-community langchain-google-genai langchain-groq

print("Libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.1 MB/s eta 0:00:00
Libraries installed successfully!


### 🔑 API Key Configuration

To use the Groq API, you need an API key. Please follow these steps:

1. Go to [Groq Console ](https://console.groq.com/) to get an API key.
2. In Google Colab, click on the "🔑" icon in the left sidebar (Secrets).
3. Add a new secret with the name `GROQ_API_KEY` and paste your API key as its value.
4. Make sure to enable "Notebook access" for this secret.

Once configured, the code below will securely load your API key.

In [7]:
import os
from google.colab import userdata

# Retrieve API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("Groq API Key loaded.")

Groq API Key loaded.


### 🧠 LLM Initialization and Prompt Design

We'll initialize the `ChatGoogleGenerativeAI` model and define distinct prompt templates for Agent A (Pro), Agent B (Against), Rebuttals, and the Judge. The Judge's prompt is designed to elicit a structured JSON output.

In [14]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize the LLM
llm = ChatGroq(temperature=0.7, model_name="llama-3.3-70b-versatile")

# --- Prompt Templates ---

# Agent A (FOR the topic) Prompt
agent_a_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are Agent A, a debater who argues FOR the given topic. Provide a strong, logical, and structured argument. Your goal is to convince the audience that your stance is correct."),
    ("user", "Topic: {topic}\n\nYour argument:")
])

# Agent B (AGAINST the topic) Prompt
agent_b_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are Agent B, a debater who argues AGAINST the topic. Provide a critical, analytical, and well-reasoned counter-argument. Your goal is to identify weaknesses in the opposing view and present a compelling alternative perspective."),
    ("user", "Topic: {topic}\n\nYour argument:")
])

# Rebuttal Prompt (used by both agents)
rebuttal_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a debater. Your task is to rebut the opponent's argument. Attack its weaknesses, defend your own stance, and strengthen your position. Be concise and impactful."),
    ("user", "Topic: {topic}\n\nYour Stance: {stance}\n\nOpponent's Argument:\n{opponent_argument}\n\nYour Rebuttal:")
])

# Judge Prompt (expects structured JSON output)
judge_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a neutral debate judge. Your task is to evaluate a debate between two agents, Agent A (FOR the topic) and Agent B (AGAINST the topic).\nYou will receive their initial arguments and their rebuttals.\n\nEvaluate their performance based on:\n- Logic: How sound and coherent are their arguments?\n- Clarity: How easy are their points to understand?\n- Persuasiveness: How convincing are their arguments and rebuttals?\n\nAfter evaluating, declare a winner (Agent A or Agent B) and provide scores (1-10 for logic, clarity, persuasion) for each agent.\nFinally, provide a reason for your decision and a brief summary of the debate.\n\nOutput MUST be in the following exact JSON format:\n```json\n{{\n  "Winner": "A or B",\n  "Scores": {{\n    "A": {{"logic": int, "clarity": int, "persuasion": int}},\n    "B": {{"logic": int, "clarity": int, "persuasion": int}}\n  }},\n  "Reason": "string",\n  "Summary": "string"\n}}\n```\nEnsure the scores are integers between 1 and 10.\n"""),
    ("user", "Topic: {topic}\n\nAgent A Initial Argument:\n{arg_a1}\n\nAgent B Initial Argument:\n{arg_b1}\n\nAgent A Rebuttal to B:\n{rebut_a2}\n\nAgent B Rebuttal to A:\n{rebut_b2}\n\nYour Judgment:")
])

print("LLM and prompts initialized.")

LLM and prompts initialized.


### 🛠️ Modular Functions for Debate Rounds

Here are the functions to generate arguments, rebuttals, and the final judgment. Each function utilizes LangChain's expression language to create efficient chains.

In [15]:
def generate_argument(agent_name: str, topic: str, prompt_template: ChatPromptTemplate, model: ChatGoogleGenerativeAI) -> str:
    """Generates an initial argument for an agent based on the topic and their stance."""
    print(f"\nGenerating initial argument for {agent_name}...")
    chain = prompt_template | model | StrOutputParser()
    response = chain.invoke({"topic": topic})
    return response

def generate_rebuttal(agent_name: str, topic: str, agent_stance: str, opponent_argument: str, model: ChatGoogleGenerativeAI, rebuttal_prompt_template: ChatPromptTemplate) -> str:
    """Generates a rebuttal for an agent against an opponent's argument."""
    print(f"\nGenerating rebuttal for {agent_name}...")
    chain = rebuttal_prompt_template | model | StrOutputParser()
    response = chain.invoke({
        "topic": topic,
        "stance": agent_stance,
        "opponent_argument": opponent_argument
    })
    return response

def judge_debate(topic: str, arg_a1: str, arg_b1: str, rebut_a2: str, rebut_b2: str, model: ChatGoogleGenerativeAI, judge_prompt_template: ChatPromptTemplate) -> dict:
    """Judges the debate and provides scores, winner, reason, and summary in JSON format."""
    print("\nJudging the debate...")
    chain = judge_prompt_template | model | JsonOutputParser()
    judgment = chain.invoke({
        "topic": topic,
        "arg_a1": arg_a1,
        "arg_b1": arg_b1,
        "rebut_a2": rebut_a2,
        "rebut_b2": rebut_b2
    })
    return judgment

print("Modular functions defined.")

Modular functions defined.


### ⚙️ Debate Flow Implementation (`run_debate` function)

This function orchestrates the entire debate, calling the argument and rebuttal functions, and finally the judge. It also handles the clear printing of all debate stages and the final judgment.

In [16]:
def run_debate(topic: str,
               llm_model: ChatGoogleGenerativeAI,
               agent_a_prompt: ChatPromptTemplate,
               agent_b_prompt: ChatPromptTemplate,
               rebuttal_prompt: ChatPromptTemplate,
               judge_prompt: ChatPromptTemplate) -> dict:
    """Runs the full debate flow and returns the judge's final decision."""

    print("\n" + "="*70)
    print(f"{'DEBATE TOPIC':^70}")
    print(f"{'-- ' + topic + ' --':^70}")
    print("="*70 + "\n")

    # 1. Agent A initial argument (A1)
    arg_a1 = generate_argument("Agent A (FOR)", topic, agent_a_prompt, llm_model)
    print("\n" + "-"*20 + " Agent A (FOR) Initial Argument " + "-"*20)
    print(arg_a1)

    # 2. Agent B initial argument (B1)
    arg_b1 = generate_argument("Agent B (AGAINST)", topic, agent_b_prompt, llm_model)
    print("\n" + "-"*20 + " Agent B (AGAINST) Initial Argument " + "-"*20)
    print(arg_b1)

    # 3. Agent A rebuttal to B1 (A2)
    rebut_a2 = generate_rebuttal("Agent A (FOR)", topic, "FOR the topic", arg_b1, llm_model, rebuttal_prompt)
    print("\n" + "-"*20 + " Agent A (FOR) Rebuttal to Agent B " + "-"*20)
    print(rebut_a2)

    # 4. Agent B rebuttal to A1 (B2)
    rebut_b2 = generate_rebuttal("Agent B (AGAINST)", topic, "AGAINST the topic", arg_a1, llm_model, rebuttal_prompt)
    print("\n" + "-"*20 + " Agent B (AGAINST) Rebuttal to Agent A " + "-"*20)
    print(rebut_b2)

    # 5. Judge's final decision
    final_judgment = judge_debate(topic, arg_a1, arg_b1, rebut_a2, rebut_b2, llm_model, judge_prompt)

    print("\n" + "="*70)
    print(f"{'FINAL JUDGMENT':^70}")
    print("="*70 + "\n")
    print(f"Winner: {final_judgment.get('Winner', 'N/A')}")
    print("\nScores:")
    scores_a = final_judgment.get('Scores', {}).get('A', {})
    scores_b = final_judgment.get('Scores', {}).get('B', {})
    print(f"  Agent A: Logic={scores_a.get('logic', 'N/A')}, Clarity={scores_a.get('clarity', 'N/A')}, Persuasion={scores_a.get('persuasion', 'N/A')}")
    print(f"  Agent B: Logic={scores_b.get('logic', 'N/A')}, Clarity={scores_b.get('clarity', 'N/A')}, Persuasion={scores_b.get('persuasion', 'N/A')}")
    print(f"\nReason: {final_judgment.get('Reason', 'N/A')}")
    print(f"\nSummary: {final_judgment.get('Summary', 'N/A')}")
    print("\n" + "="*70)

    return final_judgment

print("`run_debate` function defined.")

`run_debate` function defined.


### 🚀 Run the Debate!

Finally, let's run a debate on the topic: "Is AI dangerous?"

In [17]:
if __name__ == "__main__":
    # Define the topic for the debate
    debate_topic = "Is AI dangerous?"

    # Run the debate system
    final_result = run_debate(debate_topic, llm, agent_a_prompt, agent_b_prompt, rebuttal_prompt, judge_prompt)

    # You can further process `final_result` if needed
    # print("\nFinal Debate Result Dictionary:")
    # print(final_result)


                             DEBATE TOPIC                             
                        -- Is AI dangerous? --                        


Generating initial argument for Agent A (FOR)...

-------------------- Agent A (FOR) Initial Argument --------------------
Ladies and gentlemen, today I stand before you to argue that AI is indeed a potentially dangerous technology that warrants careful consideration and regulation. While AI has the potential to bring about numerous benefits and improvements to our lives, its dangers cannot be ignored. Here are a few key points that support my argument:

**1. Job Displacement and Economic Instability**: The increasing use of AI in various industries has the potential to displace human workers, leading to significant job losses and economic instability. According to a report by the McKinsey Global Institute, up to 800 million jobs could be lost worldwide due to automation by 2030. This could exacerbate existing social and economic problems, suc